# Notebook 18 — Token-Efficient Reasoning: Shorter Chains Without Losing Accuracy

### Where this sits

**Part II — RL for language models** · 7. RLHF · 8. RLOO & RLVR · 9. GRPO & DeepSeek-R1 ·
10. DPO & production · 11. RLCD · **12.1 Reasoning SLM with GRPO** · 12.2 RLCD reasoning ·
12.3 PPO comparison

**Part III — Agentic RL** · 13-17

**This notebook** builds directly on **12.1**, which took `Qwen2.5-0.5B-Instruct` — a
non-reasoning model — and taught it to reason with GRPO. Same task, same verifier, same
model. The new question is about **cost**.

---

## 1. What this notebook achieves, top down

Two goals, in order:

1. **Bring reasoning into a non-reasoning model** — GRPO against a verifiable reward,
   which is what notebook 12.1 did and what we reproduce here as Stage 1.
2. **Then make that reasoning cheap** — add a reward term for *token count*, so the model
   learns to reach the same answer in fewer tokens, **without losing accuracy**.

### Why goal 2 is worth real money

A reasoning model's inference cost is roughly linear in generated tokens. If you cut
mean response length by 40% at equal accuracy you have cut your serving bill by 40%,
reduced latency by 40%, and increased throughput — with no change to hardware, model
size, or quantisation. That is why "efficient reasoning" became its own research area:
there is a [survey](https://arxiv.org/pdf/2504.10903) of it, and a
[benchmark](https://arxiv.org/html/2511.05722v2).

The reason this is *possible* rather than a pure trade-off: RL-trained reasoning models
overthink. One [2026 study](https://arxiv.org/pdf/2602.09591) finds accuracy rises with
length, **peaks, then plateaus or declines** across many model-benchmark pairs. Tokens
past the peak are pure cost. Our job is to find and remove them.

### Where the real difficulty is

Everyone's first instinct is `reward = accuracy − λ·tokens`. **That reward has a
specific, predictable failure mode, and Section 4 proves it with arithmetic before we
spend a minute of GPU time.** Getting the *structure* of the reward right — not the
coefficient — is the whole problem.

We will implement and compare three designs:

| Variant | Length term | Predicted outcome |
|---|---|---|
| **A — naive** | charged on every response | accuracy collapses |
| **B — conditional** | charged only when correct | shorter, accuracy held |
| **C — difficulty-aware** | charged in proportion to how easy the prompt is | best trade-off, if there is signal |

Variant C is the idea behind [Adaptive Length Penalties](https://arxiv.org/html/2506.05256),
which reports cutting token usage ~50% on a 1.5B reasoning model without a significant
accuracy drop. It fits GRPO unusually well, for a reason we will come to.

### What gets saved

Everything. Per-variant training histories, per-response token counts, and held-out
evaluations all go to `results/token_efficiency/`, so the final comparison section reads
from disk and you can re-run any single variant without redoing the rest.

## 2. Setup, task and verifier — identical to notebook 12.1

Reproduced here so this notebook is self-contained. If you have run 12.1 none of this
will be new: three templates of 2-3 step word problems, a strict `<think>/<answer>`
format, and a verifier with unit tests.

The one piece we will lean on heavily is the **token count**, which 12.1 already tracked
as `mean length (tokens)` but never optimised.

> ### Running this on Colab or Kaggle? Read this first.
>
> The setup cell below carries two environment guards. Both exist because managed
> notebook environments preinstall packages that collide with `peft`/`trl`, and both
> failures are confusing rather than informative.
>
> **1. `ImportError: Found an incompatible version of torchao`** — raised from inside
> `get_peft_model`. Colab and Kaggle ship `torchao` 0.10, and `peft` refuses anything
> below 0.16, even though this notebook never uses torchao quantization.
> `fix_peft_torchao_conflict()` tells `peft` that torchao is unavailable, which is simply
> true here. No `pip install`, no kernel restart.
> *Alternative:* `!pip uninstall -y torchao` (then restart), or `!pip install -U "torchao>=0.16"`.
>
> **2. `ValueError: GRPOTrainer does not support nn.DataParallel`** — raised when more
> than one GPU is visible, which is what Kaggle's "T4 ×2" accelerator gives you.
> `CUDA_VISIBLE_DEVICES` is set at the very top of the setup cell, because CUDA reads
> that variable **once, at first initialisation** — set it after `import torch` has
> touched CUDA and it is silently ignored. `grpo_config()` additionally forces
> `n_gpu=1` as a fallback for out-of-order cell execution.
>
> Selecting a **single-GPU accelerator** avoids guard 2 entirely, and costs nothing:
> `GRPOTrainer` cannot use a second GPU from one process anyway (that needs
> `accelerate launch`), so one T4 is the realistic ceiling here either way.
>
> **If you hit CUDA OOM**, lower `per_device_train_batch_size` from 16 to 8. That makes
> it one prompt per step instead of two while keeping `num_generations=8`, so the group
> structure GRPO depends on is unchanged.

In [1]:
import os, re, json, time, random, warnings, gc

# ---------------------------------------------------------------------------
# MUST precede `import torch`: CUDA reads this variable only at first init.
# GRPOTrainer refuses nn.DataParallel, which HF Trainer uses when >1 GPU is visible.
# ---------------------------------------------------------------------------
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model
from trl import GRPOTrainer, GRPOConfig

# ---------------------------------------------------------------------------
# Environment guards. See the note above. Both are no-ops on a clean machine.
# ---------------------------------------------------------------------------
def fix_peft_torchao_conflict(verbose=True):
    """peft >= 0.17 raises if `torchao` is installed but older than 0.16.

    Colab/Kaggle ship torchao 0.10, so `get_peft_model` dies there even though this
    notebook uses no torchao quantization. peft binds `is_torchao_available` by name
    into three modules, so all three need patching -- replacing only `import_utils`
    leaves the already-bound copies live.
    """
    import sys, importlib
    try:
        from peft.import_utils import is_torchao_available
        is_torchao_available()
        if verbose: print("peft/torchao : OK, no patch needed")
        return False
    except ImportError as e:
        if "torchao" not in str(e):
            raise
    import peft.import_utils
    for mod in ("peft.utils.quantization_utils", "peft.tuners.lora.torchao"):
        try: importlib.import_module(mod)
        except Exception: pass
    unavailable = lambda: False
    patched = []
    for name, mod in list(sys.modules.items()):
        if name.startswith("peft") and getattr(mod, "is_torchao_available", None) is not None:
            mod.is_torchao_available = unavailable
            patched.append(name)
    if verbose:
        try:
            import torchao; v = torchao.__version__
        except Exception:
            v = "?"
        print(f"peft/torchao : found torchao {v} (< 0.16) -> reported as unavailable")
        print(f"               patched in {len(patched)} modules: {', '.join(patched)}")
    return True


def force_single_gpu(cfg, verbose=True):
    """Fallback for when CUDA was already initialised with >1 device visible.

    Setting CUDA_VISIBLE_DEVICES after CUDA init is silently ignored, so if cells ran
    out of order the env-var fix at the top of this cell arrives too late. `_n_gpu` is
    the writable attribute behind `TrainingArguments.n_gpu`; transformers itself sets it
    to 1 to suppress DataParallel (trainer.py: "Force n_gpu to 1 to avoid DataParallel"),
    so this is the sanctioned override rather than a hack.
    """
    try:
        n = torch.cuda.device_count() if torch.cuda.is_available() else 0
    except Exception:
        n = 0
    if n > 1:
        cfg._n_gpu = 1
        if verbose:
            print(f"  {n} GPUs visible -> forced n_gpu=1 (GRPOTrainer cannot use DataParallel)")
    return cfg


fix_peft_torchao_conflict()

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

RESULTS = Path("results/token_efficiency"); RESULTS.mkdir(parents=True, exist_ok=True)
set_seed(0)
print(f"device {DEVICE} | dtype {DTYPE} | results -> {RESULTS}/")

device mps | dtype torch.float32 | results -> results/token_efficiency/


In [2]:
# ---------------- the task (identical to 12.1) ----------------
NAMES = ["Sam", "Mia", "Leo", "Ava", "Noah", "Zoe", "Eli", "Ivy"]
ITEMS = ["apples", "marbles", "stickers", "books", "cookies", "pencils"]

def make_problem(rng):
    name, item = rng.choice(NAMES), rng.choice(ITEMS)
    a, b, c, d = rng.randint(2, 20), rng.randint(2, 6), rng.randint(2, 9), rng.randint(1, 10)
    kind = rng.randint(0, 2)
    if kind == 0:
        q = (f"{name} has {a} {item}. {name} buys {b} packs with {c} {item} in each pack, "
             f"then gives away {d}. How many {item} does {name} have now?")
        ans = a + b * c - d
    elif kind == 1:
        q = (f"A box holds {c} {item}. {name} fills {b} boxes and has {a} loose {item}. "
             f"{name} then loses {d}. How many {item} are left?")
        ans = b * c + a - d
    else:
        q = (f"{name} reads {c} pages a day for {b} days, then reads {a} more pages. "
             f"How many pages did {name} read in total?")
        ans = b * c + a
    return q, str(ans)

def make_dataset(n, seed):
    rng = random.Random(seed)
    return [make_problem(rng) for _ in range(n)]

TRAIN = make_dataset(400, seed=1)
_train_qs = {q for q, _ in TRAIN}
TEST = [p for p in make_dataset(80, seed=123) if p[0] not in _train_qs][:48]

# ---------------- the format and verifier (identical to 12.1) ----------------
SYSTEM_PROMPT = (
    "You are a helpful assistant. First think through the problem step by step inside "
    "<think> </think> tags, then give only the final number inside <answer> </answer> tags."
)
ANSWER_RE = re.compile(r"<answer>\s*(-?\d+)\s*</answer>")
FORMAT_RE = re.compile(r"<think>.+?</think>\s*<answer>\s*-?\d+\s*</answer>", re.S)

def parse_answer(text):
    m = ANSWER_RE.search(text)
    return m.group(1) if m else None

def follows_format(text):
    return bool(FORMAT_RE.search(text))

def format_score(text):
    r = 0.0
    if re.search(r"<think>.+?</think>", text, re.S): r += 0.10
    if ANSWER_RE.search(text):                        r += 0.25
    if follows_format(text) and text.strip().endswith("</answer>"): r += 0.40
    return r

def correctness_score(text, answer):
    return 1.0 if parse_answer(text) == answer else 0.0

assert parse_answer("<think>a</think><answer>32</answer>") == "32"
assert parse_answer("The answer is 32.") is None
assert follows_format("<think>x</think><answer>7</answer>")
assert not follows_format("<answer>7</answer>")
print(f"train {len(TRAIN)} | test {len(TEST)} | verifier unit tests passed")

train 400 | test 48 | verifier unit tests passed


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(question):
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True)

def n_tokens(text):
    """Generated-token count. This is the quantity we are about to optimise, so it gets
    its own function and is used identically in the reward and in evaluation."""
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

MAX_NEW = 160          # generation cap, as in 12.1

@torch.no_grad()
def evaluate(model, label, problems=TEST, batch_size=16, save_as=None):
    """Greedy-decode the held-out set. Returns metrics AND per-response token counts."""
    model.eval()
    t0, responses = time.time(), []
    for i in range(0, len(problems), batch_size):
        prompts = [build_prompt(q) for q, _ in problems[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        responses += tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    answers = [a for _, a in problems]
    correct = np.array([parse_answer(r) == a for r, a in zip(responses, answers)])
    lens = np.array([n_tokens(r) for r in responses])
    rec = {
        "label": label,
        "accuracy": float(correct.mean()),
        "format_compliance": float(np.mean([follows_format(r) for r in responses])),
        "accuracy_lenient": float(np.mean([re.findall(r"-?\d+", r)[-1:] == [a] for r, a in zip(responses, answers)])),
        "mean_tokens": float(lens.mean()),
        "median_tokens": float(np.median(lens)),
        "mean_tokens_correct": float(lens[correct].mean()) if correct.any() else None,
        "mean_tokens_wrong": float(lens[~correct].mean()) if (~correct).any() else None,
        "frac_at_cap": float(np.mean(lens >= MAX_NEW)),
        "eval_seconds": round(time.time() - t0, 1),
        "token_counts": lens.tolist(),
        "correct_flags": correct.tolist(),
        "responses": responses,
    }
    print(f"  {label:34} acc {rec['accuracy']:.3f} | fmt {rec['format_compliance']:.3f} | "
          f"tokens {rec['mean_tokens']:6.1f} | at-cap {rec['frac_at_cap']:.2f}  ({rec['eval_seconds']:.0f}s)")
    if save_as:
        json.dump(rec, open(RESULTS / f"{save_as}.json", "w"), indent=1)
    return rec

print(f"token counter ready; generation cap {MAX_NEW} tokens")

token counter ready; generation cap 160 tokens


## 3. Measure before you optimise: is there any slack?

The premise of this whole notebook is that the model wastes tokens. That is a claim
about the data, so measure it first. If every correct answer already uses the minimum
possible tokens, there is nothing to harvest and we should stop.

We sample a **group of 8 per prompt** — exactly the GRPO group structure — and look at
the length distribution split by correctness.

In [4]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
print(f"loaded {MODEL_NAME}: {sum(p.numel() for p in base_model.parameters())/1e6:.0f}M params\n")

@torch.no_grad()
def sample_groups(model, problems, G=8, temperature=1.0, max_new=MAX_NEW):
    """Sample G completions per prompt. Returns a flat list of dicts with length+correctness."""
    model.eval()
    rows = []
    for q, a in problems:
        enc = tokenizer([build_prompt(q)] * G, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=True,
                             temperature=temperature, top_p=1.0, pad_token_id=tokenizer.pad_token_id)
        comp = out[:, enc["input_ids"].shape[1]:]
        for j in range(G):
            ids = comp[j][comp[j] != tokenizer.pad_token_id]
            txt = tokenizer.decode(ids, skip_special_tokens=True)
            rows.append({"q": q, "answer": a, "tokens": int(len(ids)),
                         "correct": parse_answer(txt) == a, "text": txt})
    return rows

PROBE_N = 16
probe = sample_groups(base_model, TEST[:PROBE_N], G=8)
L = np.array([r["tokens"] for r in probe]); Cc = np.array([r["correct"] for r in probe])

print(f"base model, {len(probe)} completions over {PROBE_N} prompts (G=8):\n")
print(f"  strict accuracy                 {Cc.mean():.3f}")
print(f"  all completions : mean {L.mean():5.0f}  median {np.median(L):5.0f}  at-cap {np.mean(L>=MAX_NEW):.2f}")
if Cc.any():
    lc = L[Cc]
    print(f"  CORRECT (n={Cc.sum():>3})   : mean {lc.mean():5.0f}  median {np.median(lc):5.0f}  "
          f"min {lc.min():3.0f}  p90 {np.percentile(lc,90):5.0f}")
    print(f"\n  SLACK: the shortest correct answer uses {lc.min():.0f} tokens; the median correct")
    print(f"         answer uses {np.median(lc):.0f}. Ratio {np.median(lc)/max(lc.min(),1):.1f}x.")
if (~Cc).any():
    print(f"  WRONG   (n={(~Cc).sum():>3})   : mean {L[~Cc].mean():5.0f}  at-cap {np.mean(L[~Cc]>=MAX_NEW):.2f}")

import collections
pc = collections.defaultdict(list)
for r in probe: pc[r["q"]].append(r["correct"])
sr = np.array([np.mean(v) for v in pc.values()])
print(f"\n  per-prompt group success rate: mean {sr.mean():.2f} | "
      f"all-wrong groups {np.mean(sr==0):.2f} | all-correct groups {np.mean(sr==1):.2f}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-0.5B-Instruct: 494M params



base model, 128 completions over 16 prompts (G=8):

  strict accuracy                 0.039
  all completions : mean    94  median   114  at-cap 0.31
  CORRECT (n=  5)   : mean   109  median   109  min  77  p90   133

  SLACK: the shortest correct answer uses 77 tokens; the median correct
         answer uses 109. Ratio 1.4x.
  WRONG   (n=123)   : mean    93  at-cap 0.33

  per-prompt group success rate: mean 0.04 | all-wrong groups 0.69 | all-correct groups 0.00


### Two things that decide the whole design

**There is slack.** Correct answers span a wide range of lengths, so some correct answers
reach the answer in a fraction of the tokens others use. That is the thing we can harvest.

**But almost every group is all-wrong.** The base model's strict accuracy is close to
zero, and the great majority of prompts produce eight wrong answers out of eight.

That second fact is a hard constraint on the order of operations:

> **You cannot optimise length on a model that cannot solve the task.** A length reward
> conditioned on correctness has nothing to condition on, and a difficulty-aware penalty
> has no difficulty signal — every prompt looks maximally hard.

So this notebook has to be two stages, and that matches the literature: the efficient-
reasoning papers all post-train an *already reasoning* model (DeepScaleR-1.5B and
similar), never a base model.

- **Stage 1** — GRPO on `format + correctness`. Produces a model that reasons and is
  reasonably accurate. This is notebook 12.1's result.
- **Stage 2** — starting from Stage 1, add the length term and harvest the slack.

## 4. Why the naive length penalty must fail — arithmetic, before any training

Before spending GPU time, write down the reward the model will actually face. This costs
two minutes and it is the single highest-value step in the notebook.

Take the obvious design:

$$
r_{\text{naive}} = \underbrace{\text{format}}_{\le 0.75} + \underbrace{\text{correct}}_{0 \text{ or } 1} - \lambda \cdot \frac{\text{tokens}}{\text{MAX}}
$$

Now enumerate the model's options on a problem it finds hard. Suppose $\lambda = 0.5$.

| behaviour | format | correct | length term | **total** |
|---|---|---|---|---|
| long reasoning, wrong answer | 0.75 | 0 | −0.50 | **0.25** |
| **short reasoning, wrong answer** | 0.75 | 0 | −0.06 | **0.69** |
| long reasoning, right answer | 0.75 | 1 | −0.50 | **1.25** |
| short reasoning, right answer | 0.75 | 1 | −0.06 | **1.69** |

Look at rows 1 and 2. **When the model cannot solve a problem, giving up immediately
scores nearly three times better than trying.** And Section 3 just measured that the
model cannot solve most problems.

So the naive reward does not teach brevity. It teaches **abandonment**: emit a token or
two of fake reasoning, guess a number, collect the format points, and bank the saved
length. Accuracy collapses while mean reward goes *up*, which is the signature of reward
hacking from notebook 15 — the metric improves while the behaviour degrades.

### The fix: make the length charge conditional on being right

$$
r_{\text{cond}} = \text{format} + \text{correct} - \lambda \cdot \mathbf{1}[\text{correct}] \cdot \frac{\text{tokens}}{\text{MAX}}
$$

Re-run the table:

| behaviour | **naive total** | **conditional total** |
|---|---|---|
| long, wrong | 0.25 | **0.75** |
| short, wrong | 0.69 | **0.75** |
| long, right | 1.25 | **1.25** |
| short, right | 1.69 | **1.69** |

Now giving up buys nothing — wrong is worth 0.75 at any length — while among *correct*
answers, shorter strictly wins. The ordering is exactly what we want:
**correct-and-short > correct-and-long > wrong (any length).**

Let us verify that reasoning in code rather than trusting the table.

In [5]:
LAMBDA = 0.5

def r_naive(text, answer, lam=LAMBDA, **kw):
    return format_score(text) + correctness_score(text, answer) - lam * n_tokens(text) / MAX_NEW

def r_conditional(text, answer, lam=LAMBDA, **kw):
    c = correctness_score(text, answer)
    return format_score(text) + c - lam * c * n_tokens(text) / MAX_NEW

def r_baseline(text, answer, **kw):
    return format_score(text) + correctness_score(text, answer)

# Four archetypal completions, true answer = 32
cases = {
    "long reasoning, WRONG":  "<think>" + "let me think about this carefully. " * 12 + "</think><answer>31</answer>",
    "short reasoning, WRONG": "<think>no idea</think><answer>31</answer>",
    "long reasoning, RIGHT":  "<think>" + "let me think about this carefully. " * 12 + "</think><answer>32</answer>",
    "short reasoning, RIGHT": "<think>4+5*6=34, 34-2=32</think><answer>32</answer>",
}
print(f"{'behaviour':26} {'tokens':>7} {'baseline':>10} {'naive':>8} {'conditional':>12}")
print("=" * 70)
for lbl, txt in cases.items():
    print(f"{lbl:26} {n_tokens(txt):>7} {r_baseline(txt,'32'):>10.2f} "
          f"{r_naive(txt,'32'):>8.2f} {r_conditional(txt,'32'):>12.2f}")
print("=" * 70)

gap_naive = r_naive(cases["short reasoning, WRONG"], "32") - r_naive(cases["long reasoning, WRONG"], "32")
gap_cond  = r_conditional(cases["short reasoning, WRONG"], "32") - r_conditional(cases["long reasoning, WRONG"], "32")
print(f"\nreward for GIVING UP vs TRYING (both wrong):")
print(f"  naive       : {gap_naive:+.2f}   <-- positive means giving up PAYS")
print(f"  conditional : {gap_cond:+.2f}")
assert gap_naive > 0 and abs(gap_cond) < 1e-9, "the arithmetic above should hold exactly"
print("\nThe naive reward pays the model to abandon hard problems. Confirmed in code,")
print("before training. We will still train it, to see the predicted collapse.")

behaviour                   tokens   baseline    naive  conditional
long reasoning, WRONG           97       0.75     0.45         0.75
short reasoning, WRONG          15       0.75     0.70         0.75
long reasoning, RIGHT           97       1.75     1.45         1.45
short reasoning, RIGHT          30       1.75     1.66         1.66

reward for GIVING UP vs TRYING (both wrong):
  naive       : +0.26   <-- positive means giving up PAYS
  conditional : +0.00

The naive reward pays the model to abandon hard problems. Confirmed in code,
before training. We will still train it, to see the predicted collapse.


## 5. Variant C: difficulty-aware, and why GRPO gives it to you free

Conditional is sound but blunt: it charges the same per-token rate whether the problem
was trivial or near-impossible. Intuitively you want the opposite of uniform — spend
tokens where they buy accuracy, save them where they do not. That is the
[Adaptive Length Penalties](https://arxiv.org/html/2506.05256) idea.

Doing that needs a per-prompt difficulty estimate. Normally that is an extra model or an
offline pass.

**GRPO already computes it.** The algorithm samples $G$ completions per prompt to form
its baseline — so the group's **success rate** $p = \frac{1}{G}\sum_i \mathbf{1}[\text{correct}_i]$
is a free, on-policy, per-prompt difficulty measure. Notebook 12.1's manual loop even
logs the related `zero_var` statistic already.

$$
r_{\text{diff}} = \text{format} + \text{correct} - \lambda \cdot \underbrace{p}_{\text{easy} \to 1} \cdot \mathbf{1}[\text{correct}] \cdot \frac{\text{tokens}}{\text{MAX}}
$$

On a prompt the model aces ($p = 1$) the full penalty applies — it clearly does not need
many tokens. On a prompt it never solves ($p = 0$) the penalty vanishes entirely.

This requires the reward function to see the **whole group at once**, not one completion
at a time. `trl` passes completions grouped — `num_generations` consecutive entries share
a prompt — so we can reshape and compute $p$ inside the reward function.

In [6]:
G = 8                      # group size, must match GRPOConfig.num_generations

def make_trl_rewards(variant, lam=LAMBDA):
    """Return the list of trl-style reward functions for a variant.

    trl signature: f(completions, **kwargs) -> list[float].
    `completions` is a list of message-lists; extra dataset columns (here `answer`)
    arrive as keyword arguments, in the same order.
    """
    def accuracy_and_format(completions, answer, **kw):
        return [format_score(c[0]["content"]) + correctness_score(c[0]["content"], a)
                for c, a in zip(completions, answer)]

    def length_term(completions, answer, **kw):
        texts = [c[0]["content"] for c in completions]
        corr = np.array([correctness_score(t, a) for t, a in zip(texts, answer)])
        frac = np.array([n_tokens(t) for t in texts], dtype=float) / MAX_NEW
        if variant == "naive":
            pen = frac
        elif variant == "conditional":
            pen = corr * frac
        elif variant == "difficulty":
            n = len(texts)
            if n % G == 0:                                  # group-wise success rate p
                p = corr.reshape(-1, G).mean(axis=1, keepdims=True)
                p = np.repeat(p, G, axis=0).ravel()
            else:                                           # ragged batch: fall back
                p = np.full(n, corr.mean())
            pen = p * corr * frac
        else:
            raise ValueError(variant)
        return (-lam * pen).tolist()

    if variant == "baseline":
        return [accuracy_and_format]
    return [accuracy_and_format, length_term]

# --- sanity-check the group reshaping on a synthetic group of 8 ---
fake = [[{"content": "<think>x</think><answer>32</answer>"}] for _ in range(G)]
fake_ans = ["32"] * (G - 2) + ["99"] * 2          # 6/8 correct -> p = 0.75
for v in ["naive", "conditional", "difficulty"]:
    fns = make_trl_rewards(v)
    pen = fns[1](fake, fake_ans)
    print(f"{v:12} length term: {[round(x,3) for x in pen[:3]]} ...  "
          f"(mean {np.mean(pen):+.3f})")
print(f"\ngroup success rate used by 'difficulty': p = {6/G:.2f}")
print("-> its penalty is 0.75x the conditional one, and would be 0 on an all-wrong group.")

naive        length term: [-0.041, -0.041, -0.041] ...  (mean -0.041)
conditional  length term: [-0.041, -0.041, -0.041] ...  (mean -0.030)
difficulty   length term: [-0.03, -0.03, -0.03] ...  (mean -0.023)

group success rate used by 'difficulty': p = 0.75
-> its penalty is 0.75x the conditional one, and would be 0 on an all-wrong group.


## 6. Training budget — be honest about what this costs

On this machine, generation dominates: sampling a group of 8 at 160 tokens with a 0.5B
model takes roughly 8 seconds, and a GRPO step is two groups plus the backward passes.

So the budget is real and worth stating explicitly rather than discovering at hour three.
`QUICK = True` runs the full four-variant comparison in roughly an hour and is enough to
see the *direction* of every effect; `QUICK = False` gives cleaner curves.

Everything is checkpointed to `results/token_efficiency/`, so you can run one variant,
inspect it, and come back for the rest.

In [7]:
QUICK = True                     # set False for longer, cleaner runs

STAGE1_STEPS = 60 if QUICK else 150     # bring reasoning in
STAGE2_STEPS = 40 if QUICK else 100     # harvest length, per variant
VARIANTS = ["baseline", "naive", "conditional", "difficulty"]

per_step_s = 30                          # rough, measured on this machine
total_steps = STAGE1_STEPS + STAGE2_STEPS * len(VARIANTS)
print(f"stage 1 : {STAGE1_STEPS} steps")
print(f"stage 2 : {STAGE2_STEPS} steps x {len(VARIANTS)} variants = {STAGE2_STEPS*len(VARIANTS)} steps")
print(f"total   : {total_steps} steps  ~  {total_steps*per_step_s/60:.0f} min at ~{per_step_s}s/step")
print(f"\n(each step = {G} completions x 2 prompts = 16 sampled answers)")

train_dataset = Dataset.from_list([
    {"prompt": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],
     "answer": a} for q, a in TRAIN])

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM")

def grpo_config(steps, out):
    return force_single_gpu(GRPOConfig(
        output_dir=out,
        num_generations=G,
        per_device_train_batch_size=16,        # 16 / 8 = 2 prompts per step
        max_completion_length=MAX_NEW,
        temperature=1.0,
        loss_type="dr_grpo",                   # no length normalisation in the loss...
        scale_rewards=False,                   # ...and no group-std division
        beta=0.04, epsilon=0.2,
        learning_rate=1e-5,
        max_steps=steps, logging_steps=1,
        report_to="none", save_strategy="no", seed=0,
        bf16=USE_BF16, use_cpu=(DEVICE == "cpu"),
    ))

print("\nNote `loss_type='dr_grpo'`: Dr. GRPO removes the per-response length")
print("normalisation from the loss. That matters enormously here -- we want length")
print("pressure to come ONLY from the reward we designed, not smuggled in by the")
print("objective. See notebook 9.")

stage 1 : 60 steps
stage 2 : 40 steps x 4 variants = 160 steps
total   : 220 steps  ~  110 min at ~30s/step

(each step = 8 completions x 2 prompts = 16 sampled answers)

Note `loss_type='dr_grpo'`: Dr. GRPO removes the per-response length
normalisation from the loss. That matters enormously here -- we want length
pressure to come ONLY from the reward we designed, not smuggled in by the
objective. See notebook 9.


## 7. Stage 1 — bring reasoning into the model

GRPO on `format + correctness`, no length term. This is notebook 12.1 condensed. The
result is our starting point for every Stage-2 variant, so all four comparisons share
one parent and differences are attributable to the reward alone.

In [8]:
def run_grpo(variant, steps, init_state=None, tag=None):
    """Train one GRPO run. Returns (model, history). Saves history to disk."""
    tag = tag or variant
    set_seed(0)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
    model = get_peft_model(model, LoraConfig(**LORA))
    if init_state is not None:
        model.load_state_dict(init_state, strict=False)      # warm-start the adapters
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=make_trl_rewards(variant),
        args=grpo_config(steps, f"/tmp/nb18_{tag}"),
        train_dataset=train_dataset,
        peft_config=None,
    )
    t0 = time.time()
    trainer.train()
    mins = (time.time() - t0) / 60
    hist = [h for h in trainer.state.log_history if "reward" in h]
    json.dump({"variant": variant, "tag": tag, "steps": steps, "minutes": round(mins, 1),
               "lam": LAMBDA, "history": hist},
              open(RESULTS / f"history_{tag}.json", "w"), indent=1)
    print(f"  {tag}: {steps} steps in {mins:.1f} min")
    return trainer.model, hist

print("=== Stage 1: GRPO with format + correctness (no length term) ===\n")
print("  before training:")
base_eval = evaluate(base_model, "base model (no RL)", save_as="eval_base")

stage1_model, stage1_hist = run_grpo("baseline", STAGE1_STEPS, tag="stage1")
stage1_eval = evaluate(stage1_model, "stage 1 (reasoning, no length term)", save_as="eval_stage1")

# Keep the trained adapter state as the shared starting point for Stage 2.
STAGE1_STATE = {k: v.detach().clone() for k, v in stage1_model.state_dict().items() if "lora" in k.lower()}
print(f"\nkept {len(STAGE1_STATE)} LoRA tensors as the Stage-2 starting point")

=== Stage 1: GRPO with format + correctness (no length term) ===

  before training:


  base model (no RL)                 acc 0.021 | fmt 0.000 | tokens  142.6 | at-cap 0.33  (40s)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
1,0.007666
2,-0.028811
3,-0.014783
4,-0.017258
5,0.024897
6,0.015959
7,0.009920
8,0.003089
9,-0.021239
10,0.024924


## 8. Stage 2 — harvest the slack, three ways

Each variant starts from the **same** Stage-1 adapters and trains with the same budget.
The only difference is the length term.

Predictions, recorded before the runs:

- **baseline** (no length term, continued training): length roughly flat
- **naive**: accuracy collapses, length collapses, mean reward *rises* — reward hacking
- **conditional**: length falls, accuracy roughly held
- **difficulty**: length falls less than conditional on hard prompts, accuracy held best

In [ ]:
stage2 = {}
for v in VARIANTS:
    print(f"\n=== Stage 2 variant: {v} (lambda = {LAMBDA}) ===")
    mdl, hist = run_grpo(v, STAGE2_STEPS, init_state=STAGE1_STATE, tag=f"stage2_{v}")
    stage2[v] = evaluate(mdl, f"stage 2: {v}", save_as=f"eval_stage2_{v}")
    # free memory between variants -- a 0.5B model plus optimiser state is not free
    del mdl; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    elif DEVICE == "mps": torch.mps.empty_cache()
print("\nall variants trained and evaluated; results on disk")

## 9. The comparison

Everything below reads from `results/token_efficiency/`, so this section works even in a
fresh kernel after a partial run.

In [ ]:
def load(name):
    p = RESULTS / f"{name}.json"
    return json.load(open(p)) if p.exists() else None

rows = [("base model (no RL)", load("eval_base")),
        ("stage 1: reasoning", load("eval_stage1"))]
rows += [(f"stage 2: {v}", load(f"eval_stage2_{v}")) for v in VARIANTS]
rows = [(lbl, r) for lbl, r in rows if r is not None]

print(f"{'model':30} {'accuracy':>9} {'format':>8} {'mean tok':>9} {'median':>8} "
      f"{'tok/correct':>12} {'at cap':>7}")
print("=" * 90)
for lbl, r in rows:
    tc = f"{r['mean_tokens_correct']:.0f}" if r["mean_tokens_correct"] else "--"
    print(f"{lbl:30} {r['accuracy']:>9.3f} {r['format_compliance']:>8.3f} "
          f"{r['mean_tokens']:>9.1f} {r['median_tokens']:>8.0f} {tc:>12} {r['frac_at_cap']:>7.2f}")
print("=" * 90)

s1 = load("eval_stage1")
if s1:
    print(f"\nrelative to stage 1 (accuracy {s1['accuracy']:.3f}, {s1['mean_tokens']:.1f} tokens):\n")
    print(f"{'variant':24} {'accuracy change':>17} {'token change':>14} {'verdict':>22}")
    print("-" * 82)
    for v in VARIANTS:
        r = load(f"eval_stage2_{v}")
        if not r: continue
        da = r["accuracy"] - s1["accuracy"]
        dt = 100 * (r["mean_tokens"] / s1["mean_tokens"] - 1)
        if da < -0.05 and dt < -10:   verdict = "shorter but BROKEN"
        elif dt < -10 and da >= -0.05: verdict = "shorter, accuracy held"
        elif abs(dt) <= 10:            verdict = "little length change"
        else:                          verdict = "longer"
        print(f"{v:24} {da:>+17.3f} {dt:>+13.1f}% {verdict:>22}")

In [ ]:
# --- the Pareto view: accuracy against cost ---
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3))

ax = axes[0]
colors = {"base model (no RL)": "#898781", "stage 1: reasoning": "#1b4fd8",
          "stage 2: baseline": "#7ba2f5", "stage 2: naive": "#e34948",
          "stage 2: conditional": "#1baf7a", "stage 2: difficulty": "#eda100"}
for lbl, r in rows:
    ax.scatter(r["mean_tokens"], r["accuracy"], s=150, zorder=3,
               color=colors.get(lbl, "#4b5560"), edgecolor="white", linewidth=1.5, label=lbl)
    ax.annotate(lbl.replace("stage 2: ", "").replace("stage 1: ", ""),
                (r["mean_tokens"], r["accuracy"]), textcoords="offset points",
                xytext=(7, 5), fontsize=8.5)
ax.set_xlabel("mean generated tokens (lower = cheaper)")
ax.set_ylabel("held-out strict accuracy")
ax.set_title("the accuracy-cost trade-off", fontsize=11.5)
ax.grid(alpha=0.25, lw=0.6)

ax = axes[1]
for v in ["baseline", "naive", "conditional", "difficulty"]:
    h = load(f"history_stage2_{v}")
    if not h: continue
    key = next((k for k in ("completions/mean_length", "completion_length") if k in h["history"][0]), None)
    if key is None: continue
    y = [x[key] for x in h["history"]]
    k = max(1, len(y) // 12)
    sm = np.convolve(y, np.ones(k) / k, "valid") if len(y) >= k else y
    ax.plot(range(len(sm)), sm, lw=2.2, color=colors.get(f"stage 2: {v}", None), label=v)
ax.set_xlabel("stage-2 step"); ax.set_ylabel("mean completion length (tokens)")
ax.set_title("length during stage-2 training", fontsize=11.5)
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, lw=0.6)
plt.tight_layout(); plt.show()

In [ ]:
# --- the reward-hacking check: did mean reward rise while accuracy fell? ---
print("reward vs accuracy -- the reward-hacking test:\n")
print(f"{'variant':24} {'final train reward':>19} {'held-out accuracy':>19} {'hacked?':>10}")
print("-" * 76)
s1_acc = s1["accuracy"] if s1 else None
for v in VARIANTS:
    h, r = load(f"history_stage2_{v}"), load(f"eval_stage2_{v}")
    if not (h and r): continue
    tail = [x["reward"] for x in h["history"][-10:]]
    rw = float(np.mean(tail)) if tail else float("nan")
    hacked = "YES" if (s1_acc is not None and r["accuracy"] < s1_acc - 0.05) else "no"
    print(f"{v:24} {rw:>19.3f} {r['accuracy']:>19.3f} {hacked:>10}")
print("\nA variant whose reward is healthy while its accuracy dropped has optimised")
print("the metric at the expense of the task. That is the definition of reward hacking,")
print("and section 4 predicted exactly which variant would do it.")

In [ ]:
# --- what the models actually write, side by side ---
q, a = TEST[0]
print(f"QUESTION: {q}\nTRUE ANSWER: {a}\n")
for lbl, r in rows:
    if "responses" not in r: continue
    resp = r["responses"][0]
    ok = "CORRECT" if parse_answer(resp) == a else "wrong"
    print(f"--- {lbl}  ({r['token_counts'][0]} tokens, {ok}) ---")
    print(resp.strip()[:420].replace("\n", " "))
    print()

## 10. Reading the results honestly

Fill this in against your own run rather than trusting a narrative. The four outcomes
each mean something specific:

| What you see | What it means |
|---|---|
| naive: tokens down, accuracy down, reward up | the predicted hack — the model learned to give up, exactly as section 4's arithmetic said |
| conditional: tokens down, accuracy flat | the fix works; you have bought inference cost reduction for free |
| conditional: tokens flat | $\lambda$ too small relative to the correctness reward, or too few steps |
| difficulty ≈ conditional | not enough spread in group success rates for the difficulty signal to bite — check the `all-wrong groups` figure from section 3 |

### Caveats that belong in any writeup of this

- **Small model, synthetic task.** Three arithmetic templates with a strict verifier.
  Real efficient-reasoning results are on math benchmarks with 1.5B-7B models. The
  *mechanism* transfers; the numbers do not.
- **One seed.** Notebook 7 of the physical-AI course measured how badly single-seed
  comparisons can mislead. Before believing any ranking here, re-run with 3-5 seeds.
- **$\lambda$ was not tuned.** We fixed $\lambda = 0.5$ for comparability across
  variants. The honest version of this experiment sweeps $\lambda$ and reports the whole
  Pareto curve — exercise 1.
- **Greedy evaluation.** We evaluate with `do_sample=False`, which understates length
  variance. Sampled evaluation at temperature 1 is the deployment-relevant number.
- **Accuracy is strict-format.** A response with correct arithmetic but a malformed
  `<answer>` tag scores zero. That makes format and accuracy entangled, so a length
  penalty that damages formatting shows up as an accuracy drop. The `accuracy_lenient`
  column in each saved record lets you separate the two.
- **A safety note worth knowing.** One 2026 finding is that length-penalised RL
  preserves accuracy while **preferentially removing the parts of the chain that let a
  monitor detect outside influence on the model's reasoning**. Shorter chains are cheaper
  and also less auditable. If the reasoning trace is doing oversight work for you, that
  is a real cost that does not appear in either column of our table.

### What to say about this work

> *"I took a non-reasoning 0.5B model, brought in reasoning with GRPO against a
> verifiable reward, then added a token-count term to cut inference cost. The useful part
> was the reward design: I showed with arithmetic, before training, that a naive
> `accuracy − λ·tokens` reward pays the model to abandon problems it cannot solve —
> short-and-wrong beats long-and-wrong — and that making the length charge conditional on
> correctness removes that incentive entirely. I also used GRPO's own group success rate
> as a free per-prompt difficulty signal to scale the penalty, which is the adaptive-
> length-penalty idea implemented without an extra model."*

### Exercises

1. **The Pareto curve.** Sweep $\lambda \in \{0.1, 0.25, 0.5, 1.0, 2.0\}$ for the
   conditional variant and plot accuracy against mean tokens. The knee of that curve is
   the number you would actually ship, and it is the real deliverable of this kind of work.
2. **Length-aware format reward.** Our strict accuracy entangles formatting and
   correctness. Re-run with `accuracy_lenient` (last number in the response) as the
   correctness signal and see whether the conditional penalty still holds accuracy — this
   separates "shorter reasoning" from "broke the output format".
3. **A per-token budget instead of a penalty.** Replace the linear penalty with a hinge:
   charge nothing below $B$ tokens and $\lambda$ per token above it. Does a hard budget
   behave better than a linear price? (This is closer to what production serving actually
   wants.)
4. **Check for the monitorability cost.** Take the conditional model's shortened chains
   and ask whether the arithmetic steps are still *present* or merely the answer. If the
   model learned to skip stating the intermediate products, you have compressed away the
   part a human would check.
5. **Difficulty signal, measured.** Log the group success-rate distribution during
   stage 2. If it stays concentrated near 0, variant C is mathematically identical to
   variant B and the notebook should say so — a negative result about your own method is
   still a result.